In [0]:
inception = date(2011, 12, 1)
BASE = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
page_size = 100
deep_cap = 1000
USER_AGENT = "portfolio-de-project/1.0 (educational; contact: priyanka2bhutada@gmail.com)"
PREFIX = "raw/cfpb"


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7377074476887476>, line 1
----> 1 inception = date(2011, 12, 1)
      2 BASE = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
      3 page_size = 100

NameError: name 'date' is not defined

In [0]:
import requests
r = requests.get(
    "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/",
    params={"date_received_min": "2024-01-01", "date_received_max": "2024-01-07",
            "has_narrative": "true", "size": 2, "format": "json", "no_aggs": "true"},
    headers={"User-Agent": "test/1.0 (contact: you@example.com)"}, timeout=60)
print("status:", r.status_code)
print("type:", type(r.json()))
print(str(r.json())[:800])

In [0]:

def month_starts(start, end):
    curr = date(start.year, start.month, 1)
    while curr <= end: 
        if curr.month == 12:
            nextt = date(curr.year + 1, 1, 1)
        else:
            nextt = date(curr.year, curr.month + 1, 1)
        yield curr, min(nextt - timedelta(days=1), end)
        curr = nextt

In [0]:
def month_already_done(month_start):
    key = f"{PREFIX}/date={month_start:%Y-%m}/part-0000.ndjson"
    try:
        s3.head_object(Bucket=AWS_BUCKET, Key=key)
        return True
    except Exception:
        return False

In [0]:
def window_count(date_min, date_max):
    params = {
        "date_receiver_min": date_min.isoformat(),
        "date_received_max": date_max.isoformat(),
        "has_narrative":"true",
        "size":0,
        "format":"json",
    }
    r = requests.get(BASE, params=params, headers={"User-Agent": USER_AGENT}, timeout=90)
    r.raise_for_status()
    return r.json().get("hits", {}).get("total", {}).get("value", 0)

In [0]:
def page_range(date_min, date_max):
    search_after = None
    while True:
        params = {
            "date_received_min": date_min.isoformat(),
            "date_received_max": date_max.isoformat(),
            "has_narrative": "true",
            "size": page_size,
            "sort": "created_date_asc",
            "format": "json",
            "no_aggs": "true",
        }
        if search_after:
            params["search_after"] = search_after
            r = requests.get(BASE, params=params, headers={"User-Agent": USER_AGENT}, timeout=90)
            r.raise_for_status()
            hits = r.json().get("hits", {}).get("hits", [])
            if not hits:
                return
            for h in hits:
                yield h.get("_source", h)
            if len(hits) < page_size:
                return
            sort_vals = hits[-1].get("sort")
            if not sort_vals:
                return
            search_after = ",".join(str(v) for v in sort_vals)
            time.sleep(0.2)

In [0]:
def collect_range(date_min, date_max, records, seen):
    batch = []
    truncated = False
    for i, rec in enumerate(page_range(date_min, date_max)):
        if i >= deep_cap - 1:
            truncated = True
            break
        batch.append(rec)

    if truncated and date_min < date_max:
        mid = date_min + (date_max - date_min) // 2
        collect_range(date_min, mid, records, seen)
        collect_range(mid + timedelta(days=1), date_max, records, seen)
    else:
        for rec in batch:
            cid = rec.get("complaint_id")
            if cid in seen:
                continue
            seen.add(cid)
            records.append(rec)

In [0]:
import io

def land_month(month_start, month_end):
    records, seen = [], set()
    collect_range(month_start, month_end, records, seen)
    if not records:
        return 0
    buf = io.StringIO()
    for rec in records:
        buf.write(json.dumps(rec, ensure_ascii=False) + "\n")
    key = f"{PREFIX}/dt={month_start:%Y-%m}/part-0000.ndjson"
    s3.put_object(Bucket=AWS_BUCKET, Key=key, Body=buf.getvalue().encode("utf-8"))
    return len(records)

In [0]:
URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

In [0]:
import boto3, os, requests

s3 = boto3.client("s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION)

In [0]:
with requests.get(URL, stream=True, timeout=600) as r:
    r.raise_for_status()
    s3.upload_fileobj(r.raw, "credit-risk-gpt", "raw/cfpb/complaints.csv.zip")

print("landed raw/cfpb/complaints.csv.zip")

In [0]:
cfpb = (spark.read.format("csv")
        .option("header", "true")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", "false")
        .load("s3://credit-risk-gpt/raw/cfpb/complaints.csv.zip"))

In [0]:
df = spark.read.format("csv").option("header", "true").load("s3://credit-risk-gpt/raw/cfpb/")

In [0]:
CATALOG = "workspace"
SCHEMA  = "src"
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.src.cfpb

In [0]:
import os
import requests

RAW_DIR = "/Volumes/workspace/src/cfpb/raw"
ZIP_PATH = f"{RAW_DIR}/complaints.csv.zip"
URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

os.makedirs(RAW_DIR, exist_ok=True)

if os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 1e9:
    print(f"Already downloaded: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e9:.2f} GB) — skipping.")
else:
    with requests.get(URL, stream=True, timeout=60) as resp:
        resp.raise_for_status()
        done = 0
        with open(ZIP_PATH, "wb") as f:
            for chunk in resp.iter_content(chunk_size=16 * 1024 * 1024):
                f.write(chunk)
                done += len(chunk)
                print(f"\r{done/1e9:.2f} GB", end="")
    print(f"\nSaved {ZIP_PATH}")

In [0]:
import zipfile
CSV_PATH = f"{RAW_DIR}/complaints.csv"

if os.path.exists(CSV_PATH) and os.path.getsize(CSV_PATH) > 1e9:
    print(f"Already extracted: {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB) — skipping.")
else:
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)
        members = zf.namelist()
        extracted = os.path.join(RAW_DIR, members[0])
        if extracted != CSV_PATH:
            os.rename(extracted, CSV_PATH)
    print(f"Extracted to {CSV_PATH} ({os.path.getsize(CSV_PATH)/1e9:.2f} GB)")

In [0]:
with open("/Volumes/workspace/src/cfpb/raw/complaints.csv") as f:
    for _ in range(3):
        print(f.readline())

In [0]:
df = (spark.read
      .option("header", True)
      .option("multiLine", True)   
      .option("quote", '"')
      .option("escape", '"')       
      .csv(CSV_PATH))

df.printSchema()
display(df.limit(10))
n = df.count()
print(f"{n:,} rows")
assert n > 3_000_000, "Row count suspiciously low — multiLine read likely broke"